# Tutorial 6: Atlas-free 3D CNN inference

Use the same task/family/domain selectors for released checkpoints or a local standardized run. Mixed-baseline CNN weights are always the default; `variant="finetuned"` is an explicit opt-in.

In [ ]:
import matplotlib.pyplot as plt

from neurovlm import AtlasFreeCNNDataProvider
from neurovlm.atlas_free_text import (
    AtlasFreeContrastiveCollator, AtlasFreeTextEmbeddingLookup,
)
from neurovlm.runtime import load_pipeline

DOMAIN = "pubmed"  # pubmed | nilearn | neurovault
provider = AtlasFreeCNNDataProvider(domain=DOMAIN, limit=2)
rows = [provider.test[index] for index in range(len(provider.test))]
batch = AtlasFreeContrastiveCollator(
    AtlasFreeTextEmbeddingLookup.published(), (36, 45, 38)
)(rows)

## Reconstruction

Pass a batch of atlas-free volumes to the autoencoder. The returned tensor has the same shape as the input; the example also computes a simple batch MSE.

In [ ]:
autoencoder = load_pipeline(family="cnn", task="autoencoder")
reconstructed = autoencoder.reconstruct(batch["volume"])
{
    "model": autoencoder.metadata.canonical_name,
    "input_shape": tuple(batch["volume"].shape),
    "output_shape": tuple(reconstructed.shape),
    "batch_mse": float((reconstructed.cpu() - batch["volume"]).square().mean()),
}

## Contrastive retrieval

`similarity` is a brain-by-text cosine-similarity matrix. For each text query, the example returns the highest-scoring map ID and its score.

In [ ]:
contrastive = load_pipeline(
    family="cnn", task="contrastive", domain=DOMAIN
)
similarity = contrastive.similarity(batch["volume"], batch["text_embedding"]).cpu()
matches = []
for text_index, text_id in enumerate(batch["text_id"]):
    brain_index = int(similarity[:, text_index].argmax())
    matches.append({
        "query_text_id": text_id,
        "best_map_id": batch["map_id"][brain_index],
        "cosine_similarity": float(similarity[brain_index, text_index]),
    })
matches

## Text-to-brain generation

Generate one volume per text embedding. The summary confirms the concrete output shape and value range.

In [ ]:
generator = load_pipeline(
    family="cnn", task="text_to_brain", domain=DOMAIN
)
generated = generator.generate(batch["text_embedding"]).cpu()
{
    "model": generator.metadata.canonical_name,
    "text_ids": batch["text_id"],
    "output_shape": tuple(generated.shape),
    "value_range": (float(generated.min()), float(generated.max())),
}

### Inspect a generated map

A center slice gives a quick qualitative check of the generated map beside its paired target.

In [ ]:
slice_index = generated.shape[-1] // 2
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(batch["volume"][0, 0, :, :, slice_index].T, cmap="magma", origin="lower")
axes[0].set_title("Paired target")
axes[1].imshow(generated[0, 0, :, :, slice_index].T, cmap="magma", origin="lower")
axes[1].set_title("Generated from text")
for ax in axes:
    ax.axis("off")
fig.suptitle(batch["text_id"][0])
fig.tight_layout()
plt.show()

## Local runs and explicit fine-tuning

Load a standardized run with `load_pipeline(..., from_run="runs/...")`. To request released domain-fine-tuned CNN weights, add `variant="finetuned"`. MLP uses the same loader with `family="mlp"`; the original `NeuroVLM` interface remains available for legacy high-level workflows.